<a href="https://colab.research.google.com/github/abhinavmarkanda/UCS547-Accelerated-Data-Science/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import time
from numba import cuda

# Check GPU
print("GPU Available:", cuda.is_available())


# ----------------------------------
# CPU FUNCTION
# ----------------------------------
def cpu_compute(x):
    return x**2 + 3*x + 5


# ----------------------------------
# GPU KERNEL
# ----------------------------------
@cuda.jit
def gpu_compute(x, y):
    i = cuda.grid(1)
    if i < x.size:
        y[i] = x[i]*x[i] + 3*x[i] + 5


# ----------------------------------
# FLOAT32 KERNEL
# ----------------------------------
@cuda.jit
def gpu_float32(x, y):
    i = cuda.grid(1)
    if i < x.size:
        y[i] = x[i]*x[i] + 3.0*x[i] + 5.0


# ----------------------------------
# FLOAT64 KERNEL
# ----------------------------------
@cuda.jit
def gpu_float64(x, y):
    i = cuda.grid(1)
    if i < x.size:
        y[i] = x[i]*x[i] + 3.0*x[i] + 5.0


# ----------------------------------
# MAIN
# ----------------------------------
N = 5_000_000

# ---------------- CPU ----------------
x = np.random.rand(N)

start = time.time()
y_cpu = cpu_compute(x)
cpu_time = time.time() - start

print("\nCPU Time:", cpu_time)


# ---------------- GPU ----------------
y_gpu = np.zeros_like(x)

d_x = cuda.to_device(x)
d_y = cuda.to_device(y_gpu)

threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

start = time.time()
gpu_compute[blocks_per_grid, threads_per_block](d_x, d_y)
cuda.synchronize()   # IMPORTANT in Colab

y_gpu = d_y.copy_to_host()
gpu_time = time.time() - start

print("GPU Time:", gpu_time)
print("Match:", np.allclose(y_cpu, y_gpu))


# ---------------- FLOAT32 ----------------
print("\n--- FLOAT32 ---")

x32 = np.random.rand(N).astype(np.float32)
y_gpu32 = np.zeros_like(x32)

d_x32 = cuda.to_device(x32)
d_y32 = cuda.to_device(y_gpu32)

start = time.time()
gpu_float32[blocks_per_grid, threads_per_block](d_x32, d_y32)
cuda.synchronize()

y_gpu32 = d_y32.copy_to_host()
print("GPU float32 Time:", time.time() - start)


# ---------------- FLOAT64 ----------------
print("\n--- FLOAT64 ---")

x64 = np.random.rand(N).astype(np.float64)
y_gpu64 = np.zeros_like(x64)

d_x64 = cuda.to_device(x64)
d_y64 = cuda.to_device(y_gpu64)

start = time.time()
gpu_float64[blocks_per_grid, threads_per_block](d_x64, d_y64)
cuda.synchronize()

y_gpu64 = d_y64.copy_to_host()
print("GPU float64 Time:", time.time() - start)

GPU Available: True

CPU Time: 0.03630709648132324
GPU Time: 1.5489518642425537
Match: True

--- FLOAT32 ---
GPU float32 Time: 0.0772562026977539

--- FLOAT64 ---
GPU float64 Time: 0.07517313957214355


In [2]:
# ----------------------------------
# 1D HISTOGRAM BENCHMARK (COLAB READY)
# ----------------------------------

import numpy as np
import time
from numba import njit

# ----------------------------------
# PARAMETERS
# ----------------------------------
N = 1_000_000
bins = 50

# Generate random data
data = np.random.rand(N)

print("Data generated:", N, "values")


# ----------------------------------
# 1. PURE PYTHON IMPLEMENTATION
# ----------------------------------
def histogram_python(data, bins):
    hist = [0] * bins
    for x in data:
        index = int(x * bins)
        if index == bins:
            index -= 1
        hist[index] += 1
    return hist


# ----------------------------------
# 2. NUMPY IMPLEMENTATION
# ----------------------------------
def histogram_numpy(data, bins):
    hist, _ = np.histogram(data, bins=bins, range=(0, 1))
    return hist


# ----------------------------------
# 3. NUMBA IMPLEMENTATION
# ----------------------------------
@njit
def histogram_numba(data, bins):
    hist = np.zeros(bins, dtype=np.int32)
    for i in range(len(data)):
        index = int(data[i] * bins)
        if index == bins:
            index -= 1
        hist[index] += 1
    return hist


# ----------------------------------
# BENCHMARKING
# ----------------------------------

# Pure Python
start = time.time()
hist_py = histogram_python(data, bins)
time_py = time.time() - start
print("\nPure Python Time:", time_py)


# NumPy
start = time.time()
hist_np = histogram_numpy(data, bins)
time_np = time.time() - start
print("NumPy Time:", time_np)


# Numba (first run = compilation)
start = time.time()
hist_nb = histogram_numba(data, bins)
time_nb1 = time.time() - start
print("Numba Time (first run):", time_nb1)


# Numba (second run = actual speed)
start = time.time()
hist_nb = histogram_numba(data, bins)
time_nb2 = time.time() - start
print("Numba Time (second run):", time_nb2)


# ----------------------------------
# CORRECTNESS CHECK
# ----------------------------------
print("\nCorrectness Check:")
print("Python vs NumPy:", np.allclose(hist_py, hist_np))
print("NumPy vs Numba:", np.allclose(hist_np, hist_nb))

Data generated: 1000000 values

Pure Python Time: 0.5916488170623779
NumPy Time: 0.030878782272338867
Numba Time (first run): 1.7618680000305176
Numba Time (second run): 0.0028569698333740234

Correctness Check:
Python vs NumPy: True
NumPy vs Numba: True


In [3]:
# ----------------------------------
# MONTE CARLO PI (COLAB READY)
# ----------------------------------

import random
import time
from numba import njit

# ----------------------------------
# 1. PURE PYTHON FUNCTION
# ----------------------------------
def monte_carlo_pi_python(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / nsamples


# ----------------------------------
# 2. NUMBA FUNCTION
# ----------------------------------
@njit
def monte_carlo_pi_numba(nsamples):
    inside = 0
    for _ in range(nsamples):
        x = random.random()
        y = random.random()
        if x*x + y*y < 1:
            inside += 1
    return 4 * inside / nsamples


# ----------------------------------
# 3. MAIN BENCHMARK
# ----------------------------------
nsamples = 5_000_000

print("Running with", nsamples, "samples...\n")


# -------- PURE PYTHON --------
start = time.time()
pi_python = monte_carlo_pi_python(nsamples)
time_python = time.time() - start

print("Python PI:", pi_python)
print("Python Time:", time_python)


# -------- NUMBA FIRST RUN (Compilation) --------
start = time.time()
pi_numba_1 = monte_carlo_pi_numba(nsamples)
time_numba_1 = time.time() - start

print("\nNumba PI (first run):", pi_numba_1)
print("Numba Time (first run):", time_numba_1)


# -------- NUMBA SECOND RUN (Actual Speed) --------
start = time.time()
pi_numba_2 = monte_carlo_pi_numba(nsamples)
time_numba_2 = time.time() - start

print("\nNumba PI (second run):", pi_numba_2)
print("Numba Time (second run):", time_numba_2)


# ----------------------------------
# 4. SPEEDUP FACTOR
# ----------------------------------
speedup = time_python / time_numba_2
print("\nSpeedup Factor (Python / Numba):", speedup)

Running with 5000000 samples...

Python PI: 3.1421016
Python Time: 3.341676712036133

Numba PI (first run): 3.1411496
Numba Time (first run): 0.17628169059753418

Numba PI (second run): 3.141624
Numba Time (second run): 0.0571286678314209

Speedup Factor (Python / Numba): 58.49386724537278


In [4]:
# ----------------------------------
# BRIGHTNESS ADJUSTMENT (NUMBA VECTORIZE)
# ----------------------------------

import numpy as np
from numba import vectorize
import time

# ----------------------------------
# (a) VECTORIZE FUNCTION (uint8)
# ----------------------------------
@vectorize(['uint8(uint8)'], target='cpu')
def adjust_brightness(pixel_value):
    new_val = pixel_value * 1.2
    if new_val > 255:
        return 255
    else:
        return int(new_val)


# ----------------------------------
# (b) APPLY TO 10 MILLION VALUES
# ----------------------------------
N = 10_000_000
pixels = np.random.randint(0, 256, size=N, dtype=np.uint8)

print("Generated:", N, "pixels")

start = time.time()
bright_pixels = adjust_brightness(pixels)
time_cpu = time.time() - start

print("\nCPU Vectorized Time:", time_cpu)
print("Min:", bright_pixels.min(), "Max:", bright_pixels.max())


# ----------------------------------
# (c) PARALLEL VERSION (int64)
# ----------------------------------
@vectorize(['int64(int64)'], target='parallel')
def adjust_brightness_parallel(pixel_value):
    new_val = pixel_value * 1.2
    if new_val > 255:
        return 255
    else:
        return int(new_val)


# Convert to int64 for this test
pixels_int64 = pixels.astype(np.int64)

# First run (compilation)
adjust_brightness_parallel(pixels_int64)

# Second run (actual timing)
start = time.time()
bright_parallel = adjust_brightness_parallel(pixels_int64)
time_parallel = time.time() - start

print("\nParallel Vectorized Time:", time_parallel)


# ----------------------------------
# SPEED COMPARISON
# ----------------------------------
print("\nSpeedup (CPU / Parallel):", time_cpu / time_parallel)


# ----------------------------------
# (d) PASSING LIST INSTEAD OF ARRAY
# ----------------------------------
pixel_list = list(pixels[:10])  # small sample list

print("\nInput type (list):", type(pixel_list))

result_from_list = adjust_brightness(pixel_list)

print("Output type:", type(result_from_list))
print("Output values:", result_from_list)

Generated: 10000000 pixels

CPU Vectorized Time: 0.005371809005737305
Min: 0 Max: 255

Parallel Vectorized Time: 0.0296170711517334

Speedup (CPU / Parallel): 0.1813754296708339

Input type (list): <class 'list'>
Output type: <class 'numpy.ndarray'>
Output values: [ 45 255 255  52 135 124 123  66 144  94]


In [5]:
# ----------------------------------
# LOGISTIC REGRESSION (NUMPY vs NUMBA)
# ----------------------------------

import numpy as np
import time
from numba import njit

# ----------------------------------
# 1. GENERATE SYNTHETIC DATA
# ----------------------------------
np.random.seed(0)

N = 100_000   # samples
D = 10        # features

# Features
X = np.random.randn(N, D)

# True weights (for generating labels)
true_w = np.random.randn(D)

# Generate labels {-1, +1}
y = np.sign(X @ true_w)

# Replace 0 with 1 (rare case)
y[y == 0] = 1

print("Data shape:", X.shape)


# ----------------------------------
# 2. SIGMOID FUNCTION
# ----------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# ----------------------------------
# 3. NUMPY LOGISTIC REGRESSION
# ----------------------------------
def train_numpy(X, y, lr=0.01, epochs=100):
    N, D = X.shape
    w = np.zeros(D)

    for _ in range(epochs):
        z = X @ w
        yz = y * z
        grad = -(1/N) * (X.T @ (y * (1 / (1 + np.exp(yz)))))
        w -= lr * grad

    return w


# ----------------------------------
# 4. NUMBA LOGISTIC REGRESSION
# ----------------------------------
@njit
def train_numba(X, y, lr=0.01, epochs=100):
    N, D = X.shape
    w = np.zeros(D)

    for _ in range(epochs):
        grad = np.zeros(D)

        for i in range(N):
            z = 0.0
            for j in range(D):
                z += X[i, j] * w[j]

            yz = y[i] * z
            factor = -y[i] / (1.0 + np.exp(yz))

            for j in range(D):
                grad[j] += X[i, j] * factor

        for j in range(D):
            w[j] -= lr * grad[j] / N

    return w


# ----------------------------------
# 5. TRAIN + BENCHMARK
# ----------------------------------

# NumPy
start = time.time()
w_numpy = train_numpy(X, y)
time_numpy = time.time() - start

print("\nNumPy Time:", time_numpy)


# Numba (first run = compilation)
start = time.time()
w_numba = train_numba(X, y)
time_numba_1 = time.time() - start

print("Numba Time (first run):", time_numba_1)


# Numba (second run = actual speed)
start = time.time()
w_numba = train_numba(X, y)
time_numba_2 = time.time() - start

print("Numba Time (second run):", time_numba_2)


# ----------------------------------
# 6. CORRECTNESS CHECK
# ----------------------------------
print("\nWeight Difference (L2 norm):", np.linalg.norm(w_numpy - w_numba))


# ----------------------------------
# 7. SPEEDUP
# ----------------------------------
speedup = time_numpy / time_numba_2
print("Speedup (NumPy / Numba):", speedup)

Data shape: (100000, 10)

NumPy Time: 0.7862739562988281
Numba Time (first run): 1.612494707107544
Numba Time (second run): 0.6521542072296143

Weight Difference (L2 norm): 1.6667851549279532e-16
Speedup (NumPy / Numba): 1.2056564959366922


In [6]:
# ----------------------------------
# MATRIX ADDITION USING CUDA (NUMBA)
# ----------------------------------

import numpy as np
from numba import cuda
import time

# ----------------------------------
# 1. CUDA KERNEL
# ----------------------------------
@cuda.jit
def matrix_add(A, B, C):
    # 2D thread indices
    row, col = cuda.grid(2)

    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


# ----------------------------------
# 2. MATRIX SIZE
# ----------------------------------
N = 1024

# Create random matrices
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

# Output matrix
C = np.zeros((N, N), dtype=np.float32)


# ----------------------------------
# 3. COPY DATA TO GPU
# ----------------------------------
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)


# ----------------------------------
# 4. CONFIGURE THREADS & BLOCKS
# ----------------------------------
threads_per_block = (16, 16)

blocks_per_grid_x = (N + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (N + threads_per_block[1] - 1) // threads_per_block[1]

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)


# ----------------------------------
# 5. EXECUTE KERNEL
# ----------------------------------
start = time.time()

matrix_add[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()   # important

end = time.time()

print("GPU Execution Time:", end - start)


# ----------------------------------
# 6. COPY RESULT BACK
# ----------------------------------
C = d_C.copy_to_host()


# ----------------------------------
# 7. VERIFY RESULT
# ----------------------------------
C_cpu = A + B

print("Result Correct:", np.allclose(C, C_cpu))

GPU Execution Time: 0.12304234504699707
Result Correct: True
